# Task 3 — Neural Machine Translation

This notebook implements Task 3 in a linear and readable way.

It includes:

1. a reproducible 70% / 10% / 20% train-validation-test split;
2. an LSTM encoder-decoder for English → Italian;
3. a small manual hyperparameter search;
4. comparison of random, Word2Vec CBOW, and Word2Vec Skip-gram embeddings;
5. Italian → English translation using the same split;
6. a character-level English → Italian model;
7. BLEU and METEOR evaluation;
8. sentence-length and qualitative error analysis.

Attention is not included because it belongs to Task 4.

## Assumption about the folder structure

This notebook is expected to be inside:

```text
notebooks/
```

Therefore, the repository root is set explicitly with:

```python
ROOT = Path("..").resolve()
```

Run the notebook from the `notebooks` folder.

## 1. Optional package installation

In [ ]:
# Uncomment this line only if one or more packages are missing.
# %pip install numpy pandas matplotlib scikit-learn nltk gensim torch

## 2. Imports and configuration

This cell imports the required libraries, defines the fixed random seed, selects CPU or GPU, and specifies the main experiment settings.

The values are deliberately kept in one place so that the final hyperparameters are easy to report.

In [ ]:
from pathlib import Path
from collections import Counter
import copy
import gc
import json
import random
import re
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from gensim.models import Word2Vec

from nltk.translate.bleu_score import (
    corpus_bleu,
    sentence_bleu,
    SmoothingFunction,
)
from nltk.translate.meteor_score import meteor_score


# ------------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ------------------------------------------------------------------
# Project paths
# ------------------------------------------------------------------

ROOT = Path("..").resolve()

EN_PATH = ROOT / "data" / "processed" / "task2_clean.en"
IT_PATH = ROOT / "data" / "processed" / "task2_clean.it"

RESULTS_DIR = ROOT / "results" / "task3_outputs"
MODELS_DIR = ROOT / "models" / "task3"
SPLITS_DIR = ROOT / "data" / "splits"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
SPLITS_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------------
# Data and vocabulary settings
# ------------------------------------------------------------------

TEST_SIZE = 0.20
VALIDATION_SIZE = 0.10

MAX_WORD_TOKENS = 40
MIN_WORD_FREQUENCY = 2
MAX_VOCABULARY_SIZE = 25_000

# Set one of these values to an integer only for debugging on a
# smaller subset. Keep them as None for the final experiments.
MAX_TRAIN_PAIRS = None
MAX_VALIDATION_PAIRS = None
MAX_TEST_PAIRS = None


# ------------------------------------------------------------------
# Model and training settings
# ------------------------------------------------------------------

EMBEDDING_DIM = 128
BATCH_SIZE = 32
EPOCHS = 10
EARLY_STOPPING_PATIENCE = 3

TEACHER_FORCING_RATIO = 0.50
GRADIENT_CLIP = 1.0

WORD2VEC_WINDOW = 5
WORD2VEC_EPOCHS = 5


print("Device:", DEVICE)
print("Repository root:", ROOT)
print("English file:", EN_PATH)
print("Italian file:", IT_PATH)

## 3. Load the preprocessed parallel corpus

The files produced by Task 2 are read line by line. Each English line must correspond to the Italian line at the same position.

In [ ]:
def read_lines(path):
    with open(path, "r", encoding="utf-8") as file:
        return [line.rstrip("\n") for line in file]


english_sentences = read_lines(EN_PATH)
italian_sentences = read_lines(IT_PATH)

assert len(english_sentences) == len(italian_sentences), (
    "The English and Italian files do not contain the same number of lines."
)

data = pd.DataFrame(
    {
        "pair_id": np.arange(len(english_sentences)),
        "en": english_sentences,
        "it": italian_sentences,
    }
)

print(f"Aligned sentence pairs: {len(data):,}")
display(data.head())

## 4. Word tokenization and sentence-length filtering

The tokenizer separates words and punctuation while preserving their order.

Punctuation is retained because it is useful for translation and sentence generation.

Very long sentence pairs are removed because a basic encoder-decoder without attention has difficulty compressing long inputs into one fixed hidden state. The retained percentage is printed and must be reported.

In [ ]:
WORD_PATTERN = re.compile(r"\w+|[^\w\s]", flags=re.UNICODE)


def tokenize_words(text):
    return WORD_PATTERN.findall(text)


data["en_tokens"] = data["en"].map(tokenize_words)
data["it_tokens"] = data["it"].map(tokenize_words)

data["en_length"] = data["en_tokens"].map(len)
data["it_length"] = data["it_tokens"].map(len)

valid_length = (
    data["en_length"].between(1, MAX_WORD_TOKENS)
    & data["it_length"].between(1, MAX_WORD_TOKENS)
)

word_data = data.loc[valid_length].reset_index(drop=True)

retained_percentage = 100 * len(word_data) / len(data)

print(f"Pairs before length filtering: {len(data):,}")
print(f"Pairs after length filtering:  {len(word_data):,}")
print(f"Retained percentage:           {retained_percentage:.2f}%")

display(
    word_data[["en_length", "it_length"]]
    .describe(percentiles=[0.50, 0.90, 0.95, 0.99])
)

## 5. Create the train, validation, and test sets

The test set is exactly 20% of the retained data, as required.

The remaining data is divided into 70% training and 10% validation.

The same pair IDs are later reused for:

- all embedding experiments;
- English → Italian;
- Italian → English;
- the character-level experiment.

In [ ]:
train_data, test_data = train_test_split(
    word_data,
    test_size=TEST_SIZE,
    random_state=SEED,
    shuffle=True,
)

relative_validation_size = VALIDATION_SIZE / (1.0 - TEST_SIZE)

train_data, validation_data = train_test_split(
    train_data,
    test_size=relative_validation_size,
    random_state=SEED,
    shuffle=True,
)

train_data = train_data.reset_index(drop=True)
validation_data = validation_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)


def limit_rows(dataframe, maximum_rows):
    if maximum_rows is None or maximum_rows >= len(dataframe):
        return dataframe.copy()

    return (
        dataframe.sample(maximum_rows, random_state=SEED)
        .reset_index(drop=True)
    )


train_data = limit_rows(train_data, MAX_TRAIN_PAIRS)
validation_data = limit_rows(validation_data, MAX_VALIDATION_PAIRS)
test_data = limit_rows(test_data, MAX_TEST_PAIRS)


split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "pairs": [
            len(train_data),
            len(validation_data),
            len(test_data),
        ],
    }
)

display(split_summary)

train_data[["pair_id", "en", "it"]].to_csv(
    SPLITS_DIR / "task3_train.csv",
    index=False,
    encoding="utf-8",
)
validation_data[["pair_id", "en", "it"]].to_csv(
    SPLITS_DIR / "task3_validation.csv",
    index=False,
    encoding="utf-8",
)
test_data[["pair_id", "en", "it"]].to_csv(
    SPLITS_DIR / "task3_test.csv",
    index=False,
    encoding="utf-8",
)

## 6. Build the English and Italian vocabularies

The vocabularies are built only from the training data to avoid validation/test leakage.

Four special tokens are used:

- `<pad>` for padding;
- `<sos>` for the start of a sentence;
- `<eos>` for the end of a sentence;
- `<unk>` for unknown or infrequent words.

In [ ]:
SPECIAL_TOKENS = ["<pad>", "<sos>", "<eos>", "<unk>"]


class Vocabulary:
    def __init__(self, min_frequency=2, max_size=None):
        self.min_frequency = min_frequency
        self.max_size = max_size
        self.token_to_index = {}
        self.index_to_token = []

    def build(self, tokenized_sentences):
        frequencies = Counter(
            token
            for sentence in tokenized_sentences
            for token in sentence
        )

        ordered_tokens = sorted(
            frequencies.items(),
            key=lambda item: (-item[1], item[0]),
        )

        vocabulary = list(SPECIAL_TOKENS)

        for token, frequency in ordered_tokens:
            if frequency < self.min_frequency:
                continue

            if token in SPECIAL_TOKENS:
                continue

            if self.max_size is not None and len(vocabulary) >= self.max_size:
                break

            vocabulary.append(token)

        self.index_to_token = vocabulary
        self.token_to_index = {
            token: index
            for index, token in enumerate(vocabulary)
        }

        return self

    def __len__(self):
        return len(self.index_to_token)

    @property
    def pad_index(self):
        return self.token_to_index["<pad>"]

    @property
    def sos_index(self):
        return self.token_to_index["<sos>"]

    @property
    def eos_index(self):
        return self.token_to_index["<eos>"]

    @property
    def unk_index(self):
        return self.token_to_index["<unk>"]

    def encode(self, tokens):
        encoded = [
            self.token_to_index.get(token, self.unk_index)
            for token in tokens
        ]

        return [self.sos_index, *encoded, self.eos_index]

    def decode(self, indices):
        tokens = []

        for index in indices:
            token = self.index_to_token[int(index)]

            if token == "<eos>":
                break

            if token not in SPECIAL_TOKENS:
                tokens.append(token)

        return tokens


english_vocab = Vocabulary(
    min_frequency=MIN_WORD_FREQUENCY,
    max_size=MAX_VOCABULARY_SIZE,
).build(train_data["en_tokens"])

italian_vocab = Vocabulary(
    min_frequency=MIN_WORD_FREQUENCY,
    max_size=MAX_VOCABULARY_SIZE,
).build(train_data["it_tokens"])

print("English vocabulary size:", len(english_vocab))
print("Italian vocabulary size:", len(italian_vocab))

## 7. Dataset and DataLoader

Each sentence is converted from tokens to integer indices.

Inside every batch, shorter sentences are padded to the length of the longest sentence in that batch.

In [ ]:
class TranslationDataset(Dataset):
    def __init__(
        self,
        dataframe,
        source_column,
        target_column,
        source_vocab,
        target_vocab,
        tokenizer,
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.source_column = source_column
        self.target_column = target_column
        self.source_vocab = source_vocab
        self.target_vocab = target_vocab
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        source_tokens = self.tokenizer(row[self.source_column])
        target_tokens = self.tokenizer(row[self.target_column])

        return {
            "pair_id": int(row["pair_id"]),
            "source_text": row[self.source_column],
            "target_text": row[self.target_column],
            "source_ids": torch.tensor(
                self.source_vocab.encode(source_tokens),
                dtype=torch.long,
            ),
            "target_ids": torch.tensor(
                self.target_vocab.encode(target_tokens),
                dtype=torch.long,
            ),
            "source_length": len(source_tokens),
        }


def create_collate_function(source_pad_index, target_pad_index):
    def collate(batch):
        source_sequences = [item["source_ids"] for item in batch]
        target_sequences = [item["target_ids"] for item in batch]

        return {
            "pair_ids": [item["pair_id"] for item in batch],
            "source_texts": [item["source_text"] for item in batch],
            "target_texts": [item["target_text"] for item in batch],
            "source": pad_sequence(
                source_sequences,
                batch_first=True,
                padding_value=source_pad_index,
            ),
            "source_lengths": torch.tensor(
                [len(sequence) for sequence in source_sequences],
                dtype=torch.long,
            ),
            "target": pad_sequence(
                target_sequences,
                batch_first=True,
                padding_value=target_pad_index,
            ),
            "raw_source_lengths": [
                item["source_length"] for item in batch
            ],
        }

    return collate


def create_loaders(
    source_column,
    target_column,
    source_vocab,
    target_vocab,
    tokenizer,
    training_dataframe=train_data,
    validation_dataframe=validation_data,
    test_dataframe=test_data,
):
    common_arguments = {
        "source_column": source_column,
        "target_column": target_column,
        "source_vocab": source_vocab,
        "target_vocab": target_vocab,
        "tokenizer": tokenizer,
    }

    training_dataset = TranslationDataset(
        training_dataframe,
        **common_arguments,
    )
    validation_dataset = TranslationDataset(
        validation_dataframe,
        **common_arguments,
    )
    test_dataset = TranslationDataset(
        test_dataframe,
        **common_arguments,
    )

    collate_function = create_collate_function(
        source_vocab.pad_index,
        target_vocab.pad_index,
    )

    return (
        DataLoader(
            training_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            collate_fn=collate_function,
        ),
        DataLoader(
            validation_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            collate_fn=collate_function,
        ),
        DataLoader(
            test_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            collate_fn=collate_function,
        ),
    )

## 8. LSTM encoder-decoder model

The encoder reads the source sentence and returns its final hidden state and cell state.

The decoder receives these states and generates one target token at a time.

This baseline has no attention: the complete source sentence is compressed into one fixed representation.

In [ ]:
class Encoder(nn.Module):
    def __init__(
        self,
        vocabulary_size,
        embedding_dim,
        hidden_dim,
        number_of_layers,
        dropout,
        pad_index,
        embedding_weights=None,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocabulary_size,
            embedding_dim,
            padding_idx=pad_index,
        )

        if embedding_weights is not None:
            self.embedding.weight.data.copy_(embedding_weights)

        self.dropout = nn.Dropout(dropout)

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=number_of_layers,
            batch_first=True,
            dropout=dropout if number_of_layers > 1 else 0.0,
        )

    def forward(self, source, source_lengths):
        embedded = self.dropout(self.embedding(source))

        packed_source = pack_padded_sequence(
            embedded,
            source_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        _, (hidden, cell) = self.lstm(packed_source)

        return hidden, cell


class Decoder(nn.Module):
    def __init__(
        self,
        vocabulary_size,
        embedding_dim,
        hidden_dim,
        number_of_layers,
        dropout,
        pad_index,
        embedding_weights=None,
    ):
        super().__init__()

        self.vocabulary_size = vocabulary_size

        self.embedding = nn.Embedding(
            vocabulary_size,
            embedding_dim,
            padding_idx=pad_index,
        )

        if embedding_weights is not None:
            self.embedding.weight.data.copy_(embedding_weights)

        self.dropout = nn.Dropout(dropout)

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=number_of_layers,
            batch_first=True,
            dropout=dropout if number_of_layers > 1 else 0.0,
        )

        self.output_layer = nn.Linear(
            hidden_dim,
            vocabulary_size,
        )

    def forward(self, input_token, hidden, cell):
        embedded = self.dropout(
            self.embedding(input_token.unsqueeze(1))
        )

        output, (hidden, cell) = self.lstm(
            embedded,
            (hidden, cell),
        )

        logits = self.output_layer(output.squeeze(1))

        return logits, hidden, cell


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, target_pad_index):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.target_pad_index = target_pad_index

    def forward(
        self,
        source,
        source_lengths,
        target,
        teacher_forcing_ratio,
    ):
        batch_size = source.size(0)
        target_length = target.size(1)
        target_vocabulary_size = self.decoder.vocabulary_size

        outputs = torch.zeros(
            batch_size,
            target_length,
            target_vocabulary_size,
            device=source.device,
        )

        hidden, cell = self.encoder(
            source,
            source_lengths,
        )

        input_token = target[:, 0]

        for step in range(1, target_length):
            logits, hidden, cell = self.decoder(
                input_token,
                hidden,
                cell,
            )

            outputs[:, step] = logits

            predicted_token = logits.argmax(dim=1)

            use_teacher_forcing = (
                random.random() < teacher_forcing_ratio
            )

            input_token = (
                target[:, step]
                if use_teacher_forcing
                else predicted_token
            )

        return outputs


def build_model(
    source_vocab,
    target_vocab,
    hidden_dim,
    number_of_layers,
    dropout,
    source_embedding_weights=None,
    target_embedding_weights=None,
):
    encoder = Encoder(
        vocabulary_size=len(source_vocab),
        embedding_dim=EMBEDDING_DIM,
        hidden_dim=hidden_dim,
        number_of_layers=number_of_layers,
        dropout=dropout,
        pad_index=source_vocab.pad_index,
        embedding_weights=source_embedding_weights,
    )

    decoder = Decoder(
        vocabulary_size=len(target_vocab),
        embedding_dim=EMBEDDING_DIM,
        hidden_dim=hidden_dim,
        number_of_layers=number_of_layers,
        dropout=dropout,
        pad_index=target_vocab.pad_index,
        embedding_weights=target_embedding_weights,
    )

    return Seq2Seq(
        encoder,
        decoder,
        target_vocab.pad_index,
    ).to(DEVICE)

## 9. Training and validation functions

Training uses:

- cross-entropy loss;
- teacher forcing;
- Adam;
- gradient clipping;
- early stopping based on validation loss.

The best validation checkpoint is restored before test evaluation.

In [ ]:
def calculate_loss(outputs, targets, criterion):
    vocabulary_size = outputs.size(-1)

    return criterion(
        outputs[:, 1:].reshape(-1, vocabulary_size),
        targets[:, 1:].reshape(-1),
    )


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()

    total_loss = 0.0
    total_examples = 0

    for batch in loader:
        source = batch["source"].to(DEVICE)
        source_lengths = batch["source_lengths"]
        target = batch["target"].to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(
            source,
            source_lengths,
            target,
            teacher_forcing_ratio=TEACHER_FORCING_RATIO,
        )

        loss = calculate_loss(
            outputs,
            target,
            criterion,
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRADIENT_CLIP,
        )

        optimizer.step()

        total_loss += loss.item() * source.size(0)
        total_examples += source.size(0)

    return total_loss / total_examples


@torch.no_grad()
def evaluate_validation_loss(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    total_examples = 0

    for batch in loader:
        source = batch["source"].to(DEVICE)
        source_lengths = batch["source_lengths"]
        target = batch["target"].to(DEVICE)

        outputs = model(
            source,
            source_lengths,
            target,
            teacher_forcing_ratio=0.0,
        )

        loss = calculate_loss(
            outputs,
            target,
            criterion,
        )

        total_loss += loss.item() * source.size(0)
        total_examples += source.size(0)

    return total_loss / total_examples


def train_model(
    model,
    training_loader,
    validation_loader,
    learning_rate,
    checkpoint_path,
    maximum_epochs=EPOCHS,
):
    criterion = nn.CrossEntropyLoss(
        ignore_index=model.target_pad_index
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
    )

    best_validation_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0
    history = []

    start_time = time.perf_counter()

    for epoch in range(1, maximum_epochs + 1):
        training_loss = train_one_epoch(
            model,
            training_loader,
            optimizer,
            criterion,
        )

        validation_loss = evaluate_validation_loss(
            model,
            validation_loader,
            criterion,
        )

        history.append(
            {
                "epoch": epoch,
                "training_loss": training_loss,
                "validation_loss": validation_loss,
            }
        )

        print(
            f"Epoch {epoch:02d} | "
            f"train loss: {training_loss:.4f} | "
            f"validation loss: {validation_loss:.4f}"
        )

        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, checkpoint_path)
            epochs_without_improvement = 0

        else:
            epochs_without_improvement += 1

            if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
                print("Early stopping.")
                break

    training_seconds = time.perf_counter() - start_time

    model.load_state_dict(best_state)

    return (
        model,
        pd.DataFrame(history),
        best_validation_loss,
        training_seconds,
    )

## 10. Greedy decoding and evaluation metrics

The model generates the most probable token at each step until `<eos>` is produced.

Two metrics are used:

- corpus BLEU;
- mean sentence METEOR.

The METEOR implementation below avoids English-specific stemming and WordNet synonyms so that the same procedure can be applied to Italian and English.

In [ ]:
SMOOTHING = SmoothingFunction().method1


class IdentityStemmer:
    def stem(self, word):
        return word


class EmptyWordNet:
    def synsets(self, word):
        return []


IDENTITY_STEMMER = IdentityStemmer()
EMPTY_WORDNET = EmptyWordNet()


def calculate_meteor(reference_tokens, prediction_tokens):
    if not prediction_tokens:
        return 0.0

    return meteor_score(
        [reference_tokens],
        prediction_tokens,
        stemmer=IDENTITY_STEMMER,
        wordnet=EMPTY_WORDNET,
    )


@torch.no_grad()
def greedy_decode_batch(
    model,
    source,
    source_lengths,
    target_vocab,
    maximum_output_length,
):
    model.eval()

    source = source.to(DEVICE)

    hidden, cell = model.encoder(
        source,
        source_lengths,
    )

    batch_size = source.size(0)

    current_token = torch.full(
        (batch_size,),
        target_vocab.sos_index,
        dtype=torch.long,
        device=DEVICE,
    )

    generated_sequences = [
        [] for _ in range(batch_size)
    ]

    finished = torch.zeros(
        batch_size,
        dtype=torch.bool,
        device=DEVICE,
    )

    for _ in range(maximum_output_length):
        logits, hidden, cell = model.decoder(
            current_token,
            hidden,
            cell,
        )

        current_token = logits.argmax(dim=1)

        for row_index, token_index in enumerate(
            current_token.tolist()
        ):
            if finished[row_index]:
                continue

            if token_index == target_vocab.eos_index:
                finished[row_index] = True
            else:
                generated_sequences[row_index].append(
                    token_index
                )

        if finished.all():
            break

    return generated_sequences


@torch.no_grad()
def evaluate_model(
    model,
    test_loader,
    target_vocab,
    tokenizer,
    maximum_output_length,
    join_tokens_with=" ",
):
    records = []

    start_time = time.perf_counter()

    for batch in test_loader:
        generated_sequences = greedy_decode_batch(
            model,
            batch["source"],
            batch["source_lengths"],
            target_vocab,
            maximum_output_length,
        )

        for row_index, token_indices in enumerate(
            generated_sequences
        ):
            prediction_tokens = target_vocab.decode(
                token_indices
            )

            reference_tokens = tokenizer(
                batch["target_texts"][row_index]
            )

            prediction_text = join_tokens_with.join(
                prediction_tokens
            )

            local_bleu = (
                sentence_bleu(
                    [reference_tokens],
                    prediction_tokens,
                    smoothing_function=SMOOTHING,
                )
                if prediction_tokens
                else 0.0
            )

            records.append(
                {
                    "pair_id": batch["pair_ids"][row_index],
                    "source": batch["source_texts"][row_index],
                    "reference": batch["target_texts"][row_index],
                    "prediction": prediction_text,
                    "source_length": batch[
                        "raw_source_lengths"
                    ][row_index],
                    "sentence_bleu": local_bleu,
                    "meteor": calculate_meteor(
                        reference_tokens,
                        prediction_tokens,
                    ),
                    "reference_tokens": reference_tokens,
                    "prediction_tokens": prediction_tokens,
                }
            )

    inference_seconds = time.perf_counter() - start_time

    predictions = pd.DataFrame(records)

    references = [
        [tokens]
        for tokens in predictions["reference_tokens"]
    ]

    hypotheses = predictions[
        "prediction_tokens"
    ].tolist()

    bleu = 100 * corpus_bleu(
        references,
        hypotheses,
        smoothing_function=SMOOTHING,
    )

    meteor = 100 * predictions["meteor"].mean()

    return predictions, bleu, meteor, inference_seconds

## 11. Small manual hyperparameter search

To keep the tuning process clear and computationally reasonable, three configurations are compared.

Only validation loss is used to select the final configuration. The test set is not used during tuning.

In [ ]:
TUNING_CONFIGURATIONS = [
    {
        "name": "config_1",
        "hidden_dim": 256,
        "number_of_layers": 1,
        "dropout": 0.20,
        "learning_rate": 1e-3,
    },
    {
        "name": "config_2",
        "hidden_dim": 384,
        "number_of_layers": 1,
        "dropout": 0.30,
        "learning_rate": 1e-3,
    },
    {
        "name": "config_3",
        "hidden_dim": 256,
        "number_of_layers": 2,
        "dropout": 0.30,
        "learning_rate": 5e-4,
    },
]

# Tuning on a fixed training subset reduces computation while preserving
# a fair comparison between configurations.
TUNING_TRAIN_SIZE = min(30_000, len(train_data))

tuning_train_data = (
    train_data.sample(
        TUNING_TRAIN_SIZE,
        random_state=SEED,
    )
    .reset_index(drop=True)
)

tuning_validation_data = validation_data.copy()

tuning_train_loader, tuning_validation_loader, _ = create_loaders(
    source_column="en",
    target_column="it",
    source_vocab=english_vocab,
    target_vocab=italian_vocab,
    tokenizer=tokenize_words,
    training_dataframe=tuning_train_data,
    validation_dataframe=tuning_validation_data,
    test_dataframe=tuning_validation_data,
)

tuning_results = []

for configuration in TUNING_CONFIGURATIONS:
    print("\nRunning", configuration["name"])

    model = build_model(
        source_vocab=english_vocab,
        target_vocab=italian_vocab,
        hidden_dim=configuration["hidden_dim"],
        number_of_layers=configuration["number_of_layers"],
        dropout=configuration["dropout"],
    )

    model, history, best_loss, training_seconds = train_model(
        model=model,
        training_loader=tuning_train_loader,
        validation_loader=tuning_validation_loader,
        learning_rate=configuration["learning_rate"],
        checkpoint_path=(
            MODELS_DIR
            / f'{configuration["name"]}_tuning.pt'
        ),
        maximum_epochs=4,
    )

    tuning_results.append(
        {
            **configuration,
            "best_validation_loss": best_loss,
            "training_seconds": training_seconds,
        }
    )

    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


tuning_results = (
    pd.DataFrame(tuning_results)
    .sort_values("best_validation_loss")
    .reset_index(drop=True)
)

display(tuning_results)

tuning_results.to_csv(
    RESULTS_DIR / "hyperparameter_tuning.csv",
    index=False,
)

best_configuration = tuning_results.iloc[0].to_dict()

print("Selected configuration:")
display(pd.Series(best_configuration))

## 12. Prepare Word2Vec embedding matrices

Three embedding initializations are compared:

1. random embeddings;
2. Word2Vec CBOW;
3. Word2Vec Skip-gram.

Separate Word2Vec models are trained for English and Italian, using only the training data.

The embedding weights are used to initialize the encoder and decoder and remain trainable during translation training.

In [ ]:
def train_word2vec_model(tokenized_sentences, architecture):
    return Word2Vec(
        sentences=tokenized_sentences,
        vector_size=EMBEDDING_DIM,
        window=WORD2VEC_WINDOW,
        min_count=MIN_WORD_FREQUENCY,
        workers=1,
        sg=0 if architecture == "cbow" else 1,
        seed=SEED,
        epochs=WORD2VEC_EPOCHS,
    )


def create_embedding_matrix(vocab, word2vec_model):
    matrix = torch.empty(
        len(vocab),
        EMBEDDING_DIM,
    )

    torch.nn.init.normal_(
        matrix,
        mean=0.0,
        std=0.1,
    )

    matrix[vocab.pad_index].zero_()

    covered_tokens = 0

    for token, index in vocab.token_to_index.items():
        if token in SPECIAL_TOKENS:
            continue

        if token in word2vec_model.wv:
            matrix[index] = torch.tensor(
                word2vec_model.wv[token],
                dtype=torch.float32,
            )

            covered_tokens += 1

    coverage = covered_tokens / max(
        1,
        len(vocab) - len(SPECIAL_TOKENS),
    )

    return matrix, coverage


print("Training English CBOW...")
english_cbow = train_word2vec_model(
    train_data["en_tokens"].tolist(),
    "cbow",
)

print("Training Italian CBOW...")
italian_cbow = train_word2vec_model(
    train_data["it_tokens"].tolist(),
    "cbow",
)

print("Training English Skip-gram...")
english_skipgram = train_word2vec_model(
    train_data["en_tokens"].tolist(),
    "skipgram",
)

print("Training Italian Skip-gram...")
italian_skipgram = train_word2vec_model(
    train_data["it_tokens"].tolist(),
    "skipgram",
)


english_cbow_matrix, english_cbow_coverage = (
    create_embedding_matrix(
        english_vocab,
        english_cbow,
    )
)

italian_cbow_matrix, italian_cbow_coverage = (
    create_embedding_matrix(
        italian_vocab,
        italian_cbow,
    )
)

english_skipgram_matrix, english_skipgram_coverage = (
    create_embedding_matrix(
        english_vocab,
        english_skipgram,
    )
)

italian_skipgram_matrix, italian_skipgram_coverage = (
    create_embedding_matrix(
        italian_vocab,
        italian_skipgram,
    )
)

embedding_coverage = pd.DataFrame(
    {
        "embedding": ["CBOW", "Skip-gram"],
        "English coverage": [
            english_cbow_coverage,
            english_skipgram_coverage,
        ],
        "Italian coverage": [
            italian_cbow_coverage,
            italian_skipgram_coverage,
        ],
    }
)

display(embedding_coverage)

## 13. English → Italian experiments

The selected architecture is trained three times.

Only the embedding initialization changes, so the comparison isolates the effect of the embeddings.

In [ ]:
en_it_train_loader, en_it_validation_loader, en_it_test_loader = (
    create_loaders(
        source_column="en",
        target_column="it",
        source_vocab=english_vocab,
        target_vocab=italian_vocab,
        tokenizer=tokenize_words,
    )
)

embedding_setups = [
    {
        "name": "random",
        "source_weights": None,
        "target_weights": None,
        "source_coverage": np.nan,
        "target_coverage": np.nan,
    },
    {
        "name": "word2vec_cbow",
        "source_weights": english_cbow_matrix,
        "target_weights": italian_cbow_matrix,
        "source_coverage": english_cbow_coverage,
        "target_coverage": italian_cbow_coverage,
    },
    {
        "name": "word2vec_skipgram",
        "source_weights": english_skipgram_matrix,
        "target_weights": italian_skipgram_matrix,
        "source_coverage": english_skipgram_coverage,
        "target_coverage": italian_skipgram_coverage,
    },
]

all_results = []
en_it_predictions = {}
en_it_models = {}

for setup in embedding_setups:
    experiment_name = f'en_it_{setup["name"]}'

    print("\n" + "=" * 80)
    print("Running", experiment_name)

    model = build_model(
        source_vocab=english_vocab,
        target_vocab=italian_vocab,
        hidden_dim=int(best_configuration["hidden_dim"]),
        number_of_layers=int(
            best_configuration["number_of_layers"]
        ),
        dropout=float(best_configuration["dropout"]),
        source_embedding_weights=setup["source_weights"],
        target_embedding_weights=setup["target_weights"],
    )

    model, history, validation_loss, training_seconds = (
        train_model(
            model=model,
            training_loader=en_it_train_loader,
            validation_loader=en_it_validation_loader,
            learning_rate=float(
                best_configuration["learning_rate"]
            ),
            checkpoint_path=(
                MODELS_DIR / f"{experiment_name}.pt"
            ),
        )
    )

    predictions, bleu, meteor, inference_seconds = (
        evaluate_model(
            model=model,
            test_loader=en_it_test_loader,
            target_vocab=italian_vocab,
            tokenizer=tokenize_words,
            maximum_output_length=MAX_WORD_TOKENS + 10,
        )
    )

    history.to_csv(
        RESULTS_DIR / f"{experiment_name}_history.csv",
        index=False,
    )

    predictions.drop(
        columns=["reference_tokens", "prediction_tokens"]
    ).to_csv(
        RESULTS_DIR / f"{experiment_name}_predictions.csv",
        index=False,
        encoding="utf-8",
    )

    all_results.append(
        {
            "experiment": experiment_name,
            "level": "word",
            "direction": "en->it",
            "embedding": setup["name"],
            "bleu": bleu,
            "meteor": meteor,
            "validation_loss": validation_loss,
            "training_seconds": training_seconds,
            "inference_seconds": inference_seconds,
            "source_embedding_coverage": setup[
                "source_coverage"
            ],
            "target_embedding_coverage": setup[
                "target_coverage"
            ],
        }
    )

    en_it_predictions[experiment_name] = predictions
    en_it_models[experiment_name] = model


en_it_results = (
    pd.DataFrame(all_results)
    .sort_values(["bleu", "meteor"], ascending=False)
    .reset_index(drop=True)
)

display(en_it_results)

best_en_it_experiment = en_it_results.iloc[0]["experiment"]
best_embedding_name = en_it_results.iloc[0]["embedding"]

print("Best English → Italian experiment:")
print(best_en_it_experiment)

## 14. Italian → English experiment

The best embedding setup from English → Italian is reused.

The same aligned train, validation, and test pair IDs are used, but the source and target languages are exchanged.

In [ ]:
if best_embedding_name == "random":
    reverse_source_weights = None
    reverse_target_weights = None

elif best_embedding_name == "word2vec_cbow":
    reverse_source_weights = italian_cbow_matrix
    reverse_target_weights = english_cbow_matrix

else:
    reverse_source_weights = italian_skipgram_matrix
    reverse_target_weights = english_skipgram_matrix


it_en_train_loader, it_en_validation_loader, it_en_test_loader = (
    create_loaders(
        source_column="it",
        target_column="en",
        source_vocab=italian_vocab,
        target_vocab=english_vocab,
        tokenizer=tokenize_words,
    )
)

it_en_model = build_model(
    source_vocab=italian_vocab,
    target_vocab=english_vocab,
    hidden_dim=int(best_configuration["hidden_dim"]),
    number_of_layers=int(
        best_configuration["number_of_layers"]
    ),
    dropout=float(best_configuration["dropout"]),
    source_embedding_weights=reverse_source_weights,
    target_embedding_weights=reverse_target_weights,
)

it_en_model, it_en_history, it_en_validation_loss, it_en_training_seconds = (
    train_model(
        model=it_en_model,
        training_loader=it_en_train_loader,
        validation_loader=it_en_validation_loader,
        learning_rate=float(
            best_configuration["learning_rate"]
        ),
        checkpoint_path=(
            MODELS_DIR / f"it_en_{best_embedding_name}.pt"
        ),
    )
)

it_en_predictions, it_en_bleu, it_en_meteor, it_en_inference_seconds = (
    evaluate_model(
        model=it_en_model,
        test_loader=it_en_test_loader,
        target_vocab=english_vocab,
        tokenizer=tokenize_words,
        maximum_output_length=MAX_WORD_TOKENS + 10,
    )
)

all_results.append(
    {
        "experiment": f"it_en_{best_embedding_name}",
        "level": "word",
        "direction": "it->en",
        "embedding": best_embedding_name,
        "bleu": it_en_bleu,
        "meteor": it_en_meteor,
        "validation_loss": it_en_validation_loss,
        "training_seconds": it_en_training_seconds,
        "inference_seconds": it_en_inference_seconds,
        "source_embedding_coverage": np.nan,
        "target_embedding_coverage": np.nan,
    }
)

display(
    pd.DataFrame(all_results)
    .query("direction == 'it->en'")
)

## 15. Character-level English → Italian model

The same encoder-decoder architecture is reused, but individual characters replace words.

Advantages:

- very small vocabulary;
- no word-level unknown words;
- ability to compose unseen word forms.

Disadvantages:

- much longer sequences;
- weaker semantic information at each recurrent step;
- higher computational cost.

The generated characters are joined into complete strings. The strings are then tokenized with the same word tokenizer before BLEU and METEOR are calculated, making the final metrics comparable with the word-level model.

In [ ]:
MAX_CHARACTERS = 180
CHARACTER_EMBEDDING_DIM = 64
CHARACTER_HIDDEN_DIM = 256


def tokenize_characters(text):
    return list(text)


def filter_by_character_length(dataframe):
    return (
        dataframe.loc[
            dataframe["en"].str.len().between(
                1,
                MAX_CHARACTERS,
            )
            & dataframe["it"].str.len().between(
                1,
                MAX_CHARACTERS,
            )
        ]
        .reset_index(drop=True)
    )


char_train_data = filter_by_character_length(train_data)
char_validation_data = filter_by_character_length(
    validation_data
)
char_test_data = filter_by_character_length(test_data)

print("Character-model split sizes:")
print("Train:", len(char_train_data))
print("Validation:", len(char_validation_data))
print("Test:", len(char_test_data))


english_char_vocab = Vocabulary(
    min_frequency=1,
    max_size=None,
).build(
    [list(sentence) for sentence in char_train_data["en"]]
)

italian_char_vocab = Vocabulary(
    min_frequency=1,
    max_size=None,
).build(
    [list(sentence) for sentence in char_train_data["it"]]
)

print("English character vocabulary:", len(english_char_vocab))
print("Italian character vocabulary:", len(italian_char_vocab))

In [ ]:
char_train_loader, char_validation_loader, char_test_loader = (
    create_loaders(
        source_column="en",
        target_column="it",
        source_vocab=english_char_vocab,
        target_vocab=italian_char_vocab,
        tokenizer=tokenize_characters,
        training_dataframe=char_train_data,
        validation_dataframe=char_validation_data,
        test_dataframe=char_test_data,
    )
)


# The same classes can be reused, but we temporarily change the
# global embedding dimension used by build_model.
original_embedding_dim = EMBEDDING_DIM
EMBEDDING_DIM = CHARACTER_EMBEDDING_DIM

character_model = build_model(
    source_vocab=english_char_vocab,
    target_vocab=italian_char_vocab,
    hidden_dim=CHARACTER_HIDDEN_DIM,
    number_of_layers=1,
    dropout=0.20,
)

EMBEDDING_DIM = original_embedding_dim


character_model, character_history, character_validation_loss, character_training_seconds = (
    train_model(
        model=character_model,
        training_loader=char_train_loader,
        validation_loader=char_validation_loader,
        learning_rate=1e-3,
        checkpoint_path=(
            MODELS_DIR / "en_it_character.pt"
        ),
    )
)

raw_character_predictions, _, _, character_inference_seconds = (
    evaluate_model(
        model=character_model,
        test_loader=char_test_loader,
        target_vocab=italian_char_vocab,
        tokenizer=tokenize_characters,
        maximum_output_length=MAX_CHARACTERS + 20,
        join_tokens_with="",
    )
)


# Re-tokenize the reconstructed strings at word level.
character_predictions = raw_character_predictions.copy()

character_predictions["reference_tokens"] = (
    character_predictions["reference"].map(tokenize_words)
)

character_predictions["prediction_tokens"] = (
    character_predictions["prediction"].map(tokenize_words)
)

character_predictions["sentence_bleu"] = (
    character_predictions.apply(
        lambda row: (
            sentence_bleu(
                [row["reference_tokens"]],
                row["prediction_tokens"],
                smoothing_function=SMOOTHING,
            )
            if row["prediction_tokens"]
            else 0.0
        ),
        axis=1,
    )
)

character_predictions["meteor"] = (
    character_predictions.apply(
        lambda row: calculate_meteor(
            row["reference_tokens"],
            row["prediction_tokens"],
        ),
        axis=1,
    )
)

character_references = [
    [tokens]
    for tokens in character_predictions["reference_tokens"]
]

character_hypotheses = character_predictions[
    "prediction_tokens"
].tolist()

character_bleu = 100 * corpus_bleu(
    character_references,
    character_hypotheses,
    smoothing_function=SMOOTHING,
)

character_meteor = (
    100 * character_predictions["meteor"].mean()
)

all_results.append(
    {
        "experiment": "en_it_character",
        "level": "character",
        "direction": "en->it",
        "embedding": "character_embedding",
        "bleu": character_bleu,
        "meteor": character_meteor,
        "validation_loss": character_validation_loss,
        "training_seconds": character_training_seconds,
        "inference_seconds": character_inference_seconds,
        "source_embedding_coverage": np.nan,
        "target_embedding_coverage": np.nan,
    }
)

display(
    pd.DataFrame(all_results)
    .query("experiment == 'en_it_character'")
)

## 16. Fair word-level vs character-level comparison

The character model uses only test sentences that satisfy the character-length constraint.

The best word model is therefore re-evaluated on exactly the same test sentence pairs.

In [ ]:
best_word_model = en_it_models[best_en_it_experiment]

_, _, common_word_test_loader = create_loaders(
    source_column="en",
    target_column="it",
    source_vocab=english_vocab,
    target_vocab=italian_vocab,
    tokenizer=tokenize_words,
    training_dataframe=char_test_data,
    validation_dataframe=char_test_data,
    test_dataframe=char_test_data,
)

common_word_predictions, common_word_bleu, common_word_meteor, common_word_inference_seconds = (
    evaluate_model(
        model=best_word_model,
        test_loader=common_word_test_loader,
        target_vocab=italian_vocab,
        tokenizer=tokenize_words,
        maximum_output_length=MAX_WORD_TOKENS + 10,
    )
)

word_character_comparison = pd.DataFrame(
    [
        {
            "model": best_en_it_experiment,
            "bleu": common_word_bleu,
            "meteor": common_word_meteor,
            "test_pairs": len(common_word_predictions),
            "inference_seconds": common_word_inference_seconds,
        },
        {
            "model": "en_it_character",
            "bleu": character_bleu,
            "meteor": character_meteor,
            "test_pairs": len(character_predictions),
            "inference_seconds": character_inference_seconds,
        },
    ]
)

display(word_character_comparison)

word_character_comparison.to_csv(
    RESULTS_DIR / "word_character_comparison.csv",
    index=False,
)

## 17. Analyze performance as a function of sentence length

Sentence-level BLEU and METEOR are averaged inside length intervals.

These values are diagnostic. The main model comparison still uses corpus-level BLEU.

In [ ]:
best_predictions = en_it_predictions[
    best_en_it_experiment
].copy()

best_predictions["length_group"] = pd.cut(
    best_predictions["source_length"],
    bins=[0, 10, 20, 30, 40],
    labels=["1–10", "11–20", "21–30", "31–40"],
    include_lowest=True,
)

length_results = (
    best_predictions.groupby(
        "length_group",
        observed=False,
    )
    .agg(
        sentence_pairs=("pair_id", "count"),
        mean_sentence_bleu=("sentence_bleu", "mean"),
        mean_meteor=("meteor", "mean"),
    )
    .reset_index()
)

length_results["mean_sentence_bleu"] *= 100
length_results["mean_meteor"] *= 100

display(length_results)

figure, axis = plt.subplots(figsize=(7, 4))

x = np.arange(len(length_results))
width = 0.38

axis.bar(
    x - width / 2,
    length_results["mean_sentence_bleu"],
    width,
    label="Sentence BLEU",
)

axis.bar(
    x + width / 2,
    length_results["mean_meteor"],
    width,
    label="METEOR",
)

axis.set_xticks(x)
axis.set_xticklabels(
    length_results["length_group"].astype(str)
)

axis.set_xlabel("Source sentence length")
axis.set_ylabel("Mean score")
axis.set_title("Translation quality by source length")
axis.legend()
axis.grid(axis="y", alpha=0.25)

figure.tight_layout()
figure.savefig(
    RESULTS_DIR / "quality_by_length.png",
    dpi=200,
)

plt.show()

length_results.to_csv(
    RESULTS_DIR / "quality_by_length.csv",
    index=False,
)

## 18. Inspect translations and create a manual error-analysis file

The examples help identify errors that BLEU and METEOR alone cannot explain.

Suggested error categories:

- omission;
- repetition;
- rare or unknown word;
- incorrect word order;
- agreement or inflection;
- number or named-entity error;
- premature end of sentence;
- long-context failure;
- acceptable paraphrase penalized by lexical metrics.

In [ ]:
columns_to_show = [
    "source",
    "reference",
    "prediction",
    "source_length",
    "sentence_bleu",
    "meteor",
]

print("Best examples")
display(
    best_predictions.nlargest(
        3,
        "sentence_bleu",
    )[columns_to_show]
)

print("Worst examples")
display(
    best_predictions.nsmallest(
        3,
        "sentence_bleu",
    )[columns_to_show]
)

print("Longest examples")
display(
    best_predictions.nlargest(
        3,
        "source_length",
    )[columns_to_show]
)


manual_error_analysis = (
    best_predictions.nsmallest(
        20,
        "sentence_bleu",
    )
    [
        [
            "pair_id",
            "source",
            "reference",
            "prediction",
            "source_length",
            "sentence_bleu",
            "meteor",
        ]
    ]
    .copy()
)

manual_error_analysis["error_categories"] = ""
manual_error_analysis["comment"] = ""

manual_error_analysis.to_csv(
    RESULTS_DIR / "manual_error_analysis.csv",
    index=False,
    encoding="utf-8",
)

display(manual_error_analysis.head())

## 19. Save the final consolidated results

The final table contains:

- embedding comparison;
- translation-direction comparison;
- word vs character comparison;
- validation loss;
- training time;
- inference time.

In [ ]:
final_results = (
    pd.DataFrame(all_results)
    .sort_values(
        ["direction", "level", "bleu"],
        ascending=[True, True, False],
    )
    .reset_index(drop=True)
)

display(final_results)

final_results.to_csv(
    RESULTS_DIR / "task3_results.csv",
    index=False,
)

configuration_to_save = {
    "seed": SEED,
    "device": str(DEVICE),
    "train_fraction": 0.70,
    "validation_fraction": VALIDATION_SIZE,
    "test_fraction": TEST_SIZE,
    "max_word_tokens": MAX_WORD_TOKENS,
    "retained_percentage_after_length_filter": (
        retained_percentage
    ),
    "minimum_word_frequency": MIN_WORD_FREQUENCY,
    "maximum_vocabulary_size": MAX_VOCABULARY_SIZE,
    "embedding_dimension": EMBEDDING_DIM,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "teacher_forcing_ratio": TEACHER_FORCING_RATIO,
    "gradient_clip": GRADIENT_CLIP,
    "selected_configuration": {
        "hidden_dim": int(
            best_configuration["hidden_dim"]
        ),
        "number_of_layers": int(
            best_configuration["number_of_layers"]
        ),
        "dropout": float(
            best_configuration["dropout"]
        ),
        "learning_rate": float(
            best_configuration["learning_rate"]
        ),
    },
    "word2vec_window": WORD2VEC_WINDOW,
    "word2vec_epochs": WORD2VEC_EPOCHS,
    "maximum_characters": MAX_CHARACTERS,
}

with open(
    RESULTS_DIR / "task3_configuration.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        configuration_to_save,
        file,
        indent=2,
    )

print("Results saved in:", RESULTS_DIR)
print("Models saved in:", MODELS_DIR)

## Critical discussion for the report

### Architecture

The LSTM encoder-decoder is a natural recurrent baseline because it preserves token order and supports variable-length sequences. A one-layer architecture is simpler and less computationally expensive, while deeper or wider alternatives are tested during tuning. The main weakness is the fixed-vector bottleneck: the final encoder state must represent the complete source sentence.

### Hyperparameter tuning

A small manual search is preferred to a large grid search because sequence-to-sequence training is expensive. Hidden size, number of layers, dropout, and learning rate are compared. The final setup is selected only from validation loss.

### Embeddings

Random initialization measures how much lexical structure the translation objective learns alone. CBOW predicts a word from its context and is generally efficient for frequent words. Skip-gram predicts surrounding words and can represent rarer words more effectively. Their effect is measured while keeping the translation architecture and data split fixed.

### Translation direction

English → Italian and Italian → English use the same aligned split. Differences can therefore be linked to target-language morphology, vocabulary size, word order, and generation difficulty rather than different test data.

### Character model

The character model eliminates word-level OOV tokens and may better represent Italian inflection. However, character sequences are much longer and individual characters contain limited semantic information. The model is therefore expected to require more recurrent steps and training time.

### Sentence length

If scores decrease for longer sentences, this supports the fixed-vector bottleneck explanation. Short and formulaic parliamentary phrases are likely to be easier, while long clauses, rare vocabulary, named entities, numbers, and morphology can create failures.

### Metrics

BLEU provides the main corpus-level comparison. METEOR adds a complementary partial-match perspective. Neither metric fully captures semantic adequacy, so qualitative examples and manual error analysis are necessary.

## Final checklist

- [ ] The test set is exactly 20%.
- [ ] The vocabulary is built only on the training set.
- [ ] The retained percentage after length filtering is reported.
- [ ] Hyperparameters are selected using validation loss.
- [ ] Random, CBOW, and Skip-gram embeddings are compared.
- [ ] EN→IT and IT→EN use the same aligned split.
- [ ] The character model is evaluated on reconstructed text.
- [ ] Word and character models are compared on the same test pairs.
- [ ] Training and inference time are reported separately.
- [ ] Length effects and concrete translation errors are discussed.
- [ ] Attention remains separate for Task 4.